In [60]:
import numpy as np
import pandas as pd

from preprocessing_utils import ensure_nltk_resources, preprocess_texts
ensure_nltk_resources()

from metrics_utils import *

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize, LabelEncoder

from sklearn.cluster import KMeans
import skfuzzy as fuzz

seed = 42
np.random.seed(seed)

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/bernardod/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/bernardod/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [61]:
df = pd.read_json("../datasets/fixed_dataset.json")

def get_first_label(category: str) -> str:
    return str(category).split(",")[0].strip()

df["category"] = df["category"].apply(get_first_label)
label_encoder = LabelEncoder()
y_true_encoded = label_encoder.fit_transform(df["category"].apply(get_first_label))

In [62]:
CONFIG = {
    # Datasets can have different column names
    "title_column": "title",
    "abstract_column": "abstract",

    # We can change this in the future for BERT for example
    "vectorizer": "tfidf",
    
    "tfidf": dict(min_df=3, max_df=0.7, max_features=3000, ngram_range=(1, 3)),
    
    "use_svd": True,
    "svd_components": [20, 25, 35, 50, 75],  # grid
    
    "K_values": [2, 3, 4, 6, 8, 10, 12, 15],

    "algorithms": ["fcm", "kmeans"],

}

In [63]:
def tfidf_vectorize(texts):
    vectorizer = TfidfVectorizer(
        ngram_range=CONFIG["tfidf"]["ngram_range"],
        stop_words="english",
        min_df=CONFIG["tfidf"]["min_df"],
        max_df=CONFIG["tfidf"]["max_df"],
        max_features=CONFIG["tfidf"]["max_features"],
    )
    return vectorizer.fit_transform(texts)

In [64]:
texts = (df[CONFIG["title_column"]] + " " + df[CONFIG["abstract_column"]]).tolist()
texts_clean = preprocess_texts(texts, use_pos=True)

VECTORIZERS = {
    "tfidf": tfidf_vectorize,
}

X_base = VECTORIZERS[CONFIG["vectorizer"]](texts_clean)

In [65]:
def build_X(X_base, n_comp=None, use_svd=True):
    """
    Returns final X that all algorithms will use.
    - If use_svd=False: returns dense array (Assumes it's already normalized).
    - Se use_svd=True: apply SVD with n_comp.
    """
    if use_svd:
        svd = TruncatedSVD(n_components=n_comp, random_state=seed)
        X = svd.fit_transform(X_base)
        X = normalize(X, norm="l2")
    else:
        X = X_base.toarray() if hasattr(X_base, "toarray") else np.asarray(X_base)

    return X

In [66]:
def cluster_fcm(X, K, m=1.7, error=0.005, maxiter=1000):
    cntr, U, *_rest, fpc = fuzz.cluster.cmeans(
        X.T, c=K, m=m, error=error, maxiter=maxiter,
        metric="cosine", seed=seed
    )
    U = U.T
    labels = U.argmax(axis=1)
    return labels, {"fpc": float(fpc), "U": U}

def cluster_kmeans(X, K, max_iter=300):
    model = KMeans(n_clusters=K, random_state=seed, init="k-means++", max_iter=max_iter)
    labels = model.fit_predict(X)
    return labels, {"inertia": float(model.inertia_)}


In [67]:
# For cmeans only
def diagnose_collapse(U):
    eps = 1e-12
    K = U.shape[1]
    entropy = -np.sum(U * np.log(U + eps), axis=1)
    entropy_norm = float(np.mean(entropy) / np.log(K))
    avg_max_memb = float(U.max(axis=1).mean())
    return {
        "collapsed": entropy_norm > 0.85,
        "entropy_norm": entropy_norm,
        "avg_max_memb": avg_max_memb,
    }

def evaluate_all(X, y_true, y_pred):
    return {
        "ARI": calculate_ari(y_true, y_pred),
        "NMI": calculate_nmi(y_true, y_pred),
        "ACC": calculate_accuracy(y_true, y_pred),
        "SIL": calculate_silhouette(X, y_pred),
    }

In [71]:
def append_result(alg_name, y_pred, extras):
    metrics = evaluate_all(X, y_true_encoded, y_pred)
    row = {**base_row, "alg": alg_name, **metrics, **extras}
    rows.append(row)


rows = []

if CONFIG["use_svd"]:
    n_comp_list = CONFIG["svd_components"]
else:
    n_comp_list = [None]

for n_comp in n_comp_list:
    X = build_X(X_base, n_comp=n_comp, use_svd=CONFIG["use_svd"])

    for K in CONFIG["K_values"]:

        base_row = {
            "vectorizer": CONFIG["vectorizer"],
            "use_svd": CONFIG["use_svd"],
            "n_comp": n_comp,
            "K": K,
            "collapsed": False,
            "entropy_norm": np.nan,
            "avg_max_memb": np.nan,
        }
        
        if "fcm" in CONFIG["algorithms"]:
            y_pred, extra = cluster_fcm(X, K)

            collapse = diagnose_collapse(extra["U"])
            extras = {
                **collapse,
                "fpc": extra["fpc"],
            }
            append_result("FCM", y_pred, extras)
            
        if "kmeans" in CONFIG["algorithms"]:
            y_pred, extra = cluster_kmeans(X, K)
            append_result("KMeans", y_pred, {
                "inertia": extra["inertia"]
            })

results_all = pd.DataFrame(rows)
results_all.sort_values(by=["NMI", "ARI", "ACC"], ascending=False, inplace=True)
results_all.reset_index(drop=True, inplace=True)

In [72]:
filtered = results_all.copy()
filtered = filtered[~((filtered["alg"] == "FCM") & (filtered["collapsed"] == True))]

top_per_alg = (
    filtered
    .sort_values(["NMI","ARI","ACC"], ascending=False)
    .groupby("alg")
    .head(5)
)

top_per_alg


,vectorizer,use_svd,n_comp,K,collapsed,entropy_norm,avg_max_memb,alg,ARI,NMI,ACC,SIL,fpc,inertia
1,tfidf,True,20,15,False,0.491371,0.621235,FCM,0.043292,0.280184,0.289256,0.259738,0.514761,NaN
2,tfidf,True,25,15,False,0.571186,0.551873,FCM,0.028368,0.260114,0.247934,0.225871,0.444901,NaN
4,tfidf,True,75,15,False,NaN,NaN,KMeans,0.022252,0.254498,0.239669,0.074707,NaN,83.217777
5,tfidf,True,50,15,False,NaN,NaN,KMeans,0.029578,0.251680,0.247934,0.127695,NaN,73.998767
6,tfidf,True,20,12,False,0.562445,0.574699,FCM,0.039689,0.250193,0.264463,0.201955,0.462845,NaN
7,tfidf,True,20,15,False,NaN,NaN,KMeans,0.017547,0.246373,0.223140,0.259823,NaN,46.686431
8,tfidf,True,25,15,False,NaN,NaN,KMeans,0.032870,0.246037,0.256198,0.235080,NaN,54.332443
10,tfidf,True,25,12,False,0.604298,0.538345,FCM,0.039341,0.242567,0.256198,0.187916,0.429358,NaN
13,tfidf,True,50,12,False,NaN,NaN,KMeans,0.026569,0.230156,0.264463,0.111909,NaN,78.801567
14,tfidf,True,20,10,False,0.593828,0.559755,FCM,0.031661,0.229949,0.280992,0.184276,0.449835,NaN


In [70]:
results_all.query("n_comp == 35 and K == 8")

,vectorizer,use_svd,n_comp,K,collapsed,entropy_norm,avg_max_memb,alg,ARI,NMI,ACC,SIL,fpc,inertia
39,tfidf,True,35,8,True,0.999891,0.128642,FCM,0.024347,0.163946,0.280992,0.005895,0.125056,NaN
40,tfidf,True,35,8,False,NaN,NaN,KMeans,0.025381,0.158064,0.297521,0.114439,NaN,79.130506
